In [ ]:
import os
import re
import scanpy as sc
import pandas as pd
import numpy as np

In [1]:
# ==============================================================================
# 10x Visium Data Preparation for NMF and Correlation Analysis
# Customized for specific Sample and Organ lists
# ==============================================================================
# Data Directory
base_dir = '/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/raw/Samples'
samples_base_dir = '/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/raw/Samples'

# 2. ESTIMATE 结果文件夹路径（根据你的文件夹结构：processed/tumor_purity_estimate/样本名/）
estimate_base_dir = '/group/jshandl-g00/Spatial-MetScore/Spatial-MetScore/data/processed/tumor_purity_estimate'

# Defined Lists
Primary_list = ["PT-1A", "PT-2A", "PT-3A", "PT-3_PRI2", "PT-4A", "PT-5A", "PT-6A", "PT-6_PRI2", "PT-6_PRI3", "PT-7A", "PT-8A", "PT-8_PRI2", "PT-9A", "PT-10A", "PT-11A", "PT-12A", "PT-13A", "PT-13_PRI2", "PT-13_PRI3"]
Lung_list = ["PT-2B", "PT-4C", "PT-10C", "PT-11C", "PT-12C", "PT-13C"]
Liver_list = ["PT-3B", "PT-4B", "PT-5B", "PT-5C", "PT-6B", "PT-7B", "PT-7C", "PT-8B", "PT-8C", "PT-9B", "PT-9C", "PT-10B", "PT-11B", "PT-12B", "PT-13B"]
Peritoneal_list = ["PT-1C", "PT-2C", "PT-3C", "PT-4D", "PT-6C", "PT-6D", "PT-7D", "PT-8D", "PT-9D", "PT-10D", "PT-11D", "PT-12D", "PT-13D", "PT-13E"]

In [ ]:
# Create a mapping dictionary for O(1) lookup
organ_mapping = {}
for s in Primary_list: organ_mapping[s] = 'Primary'
for s in Lung_list: organ_mapping[s] = 'Lung'
for s in Liver_list: organ_mapping[s] = 'Liver'
for s in Peritoneal_list: organ_mapping[s] = 'Peritoneal'

In [ ]:
all_spots_info = []
all_adata = []

In [ ]:
def extract_patient_id(sample_name):
    """Extracts the base Patient ID (e.g., PT-13 from PT-13_PRI2 or PT-13A)."""
    match = re.match(r'(PT-\d+)', sample_name)
    return match.group(1) if match else sample_name

In [ ]:
def process_visium_samples(base_dir, estimate_csv_path=None):
    
    
    samples_dir = os.path.join(base_dir, 'Samples')
    
    if not os.path.exists(samples_dir):
        print(f"Warning: Directory {samples_dir} not found. (Assuming this is a dry run)")
        # For demonstration purposes, we will not raise an error here if folder is missing
        return pd.DataFrame(), pd.DataFrame()

    for sample_name in os.listdir(samples_dir):
        if sample_name not in organ_mapping:
            continue # Skip files/folders not in your specific list
            
        sample_path = os.path.join(samples_dir, sample_name)
        h5_path = os.path.join(sample_path, 'filtered_feature_bc_matrix.h5')
        
        if not os.path.exists(h5_path):
            print(f"Skipping {sample_name} - h5 file not found.")
            continue
            
        print(f"Processing {sample_name}...")
        
        # 1. Extract Patient Source and Organ
        patient_id = extract_patient_id(sample_name)
        organ_name = organ_mapping.get(sample_name, "Unknown")
        
        # 2. Load 10x Visium data
        adata = sc.read_10x_h5(h5_path)
        adata.var_names_make_unique()
        
        # Make spot IDs unique across different samples
        adata.obs.index = adata.obs.index + '_' + sample_name
        
        # 3. Calculate Reads (Total UMIs per spot)
        adata.obs['Reads'] = adata.X.sum(axis=1)
        
        # 4. Calculate MET Score (Example using single gene 'MET')
        if 'MET' in adata.var_names:
            met_expr = adata[:, 'MET'].X.toarray().flatten() if hasattr(adata.X, 'toarray') else adata[:, 'MET'].X.flatten()
            adata.obs['MET_Score'] = met_expr
        else:
            adata.obs['MET_Score'] = np.nan
            
        # 5. Prepare base info dataframe
        sample_info = pd.DataFrame({
            'Spot_ID': adata.obs.index,
            'Patient_Source': patient_id,
            'Sample_Name': sample_name,
            'Organ': organ_name,
            'Reads': adata.obs['Reads'],
            'MET_Score': adata.obs['MET_Score']
        })
        
        all_spots_info.append(sample_info)
        all_adata.append(adata)
        
    if not all_spots_info:
        print("No valid samples were processed. Please check your directory structure.")
        return pd.DataFrame(), pd.DataFrame()

    # Combine all metadata
    final_info_df = pd.concat(all_spots_info, ignore_index=True)
    
    # 6. Merge with ESTIMATE Scores
    if estimate_csv_path and os.path.exists(estimate_csv_path):
        print(f"Merging ESTIMATE scores from {estimate_csv_path}...")
        estimate_df = pd.read_csv(estimate_csv_path)
        final_info_df = pd.merge(final_info_df, estimate_df, on='Spot_ID', how='left')
    else:
        print("No ESTIMATE score file provided. Filling with NaNs...")
        final_info_df['ESTIMATE_TumorPurity'] = np.nan
        final_info_df['ESTIMATE_ImmuneScore'] = np.nan
        final_info_df['ESTIMATE_StromaScore'] = np.nan
        
    final_info_df.to_csv('Visium_AllSpots_InfoTable_Final.csv', index=False)
    print("-> Saved Info Table: Visium_AllSpots_InfoTable_Final.csv")
    
    # ==============================================================================
    # 7. Create the Combined Gene Expression Matrix for NMF
    # ==============================================================================
    print("Combining gene expression matrices...")
    if len(all_adata) > 1:
        combined_adata = all_adata[0].concatenate(all_adata[1:], join='inner', batch_key=None)
    else:
        combined_adata = all_adata[0]
        
    if hasattr(combined_adata.X, 'toarray'):
        expr_matrix = combined_adata.X.toarray()
    else:
        expr_matrix = combined_adata.X
        
    expr_df = pd.DataFrame(
        expr_matrix,
        index=combined_adata.obs.index,
        columns=combined_adata.var_names
    ).T  
    
    expr_df.index.name = 'Gene'
    expr_df.to_csv('Visium_Combined_ExpressionMatrix_Final.csv')
    print("-> Saved Expression Matrix: Visium_Combined_ExpressionMatrix_Final.csv")
    
    return final_info_df, expr_df

if __name__ == "__main__":
    base_dir = './raw'
    # Run the pipeline
    info_table, expr_table = process_visium_samples(
        base_dir=base_dir, 
        estimate_csv_path=None 
    )